In [11]:
import numpy as np
from scipy import signal
from scipy.io import loadmat
from scipy.linalg import eigh, inv, sqrtm
import os
import mne
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, pairwise_kernels
from sklearn.base import BaseEstimator, TransformerMixin
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning) # Ignore RuntimeWarnings from MNE
warnings.filterwarnings("ignore", category=FutureWarning) # Ignore FutureWarning from dependencies
# Ignore specific MNE deprecation warning about units
warnings.filterwarnings("ignore", message=".*unit 'V' is ignored.*")


# --- Data Loading and Preprocessing (Aligned with Paper) ---
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772} # Left Hand, Right Hand, Feet, Tongue
active_lr_event_ids = {'769': 769, '770': 770} # Left Hand, Right Hand
unknown_event_id = {'783': 783} # Cue onset during evaluation

# !!! IMPORTANT: Set the correct path to your data directory !!!
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'
true_labels_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels'
# --- FAKE PATHS FOR EXECUTION ---

os.makedirs(data_dir, exist_ok=True)
os.makedirs(true_labels_dir, exist_ok=True)
print(f"NOTE: Using placeholder directories: {data_dir}, {true_labels_dir}")
print("Please ensure the actual BCI Competition IV 2a GDF and true label MAT files are present in these locations.")
# Create dummy files if they don't exist for placeholder execution
for subj in range(1, 10):
    for phase in ['T', 'E']:
        fname_gdf = os.path.join(data_dir, f'A{subj:02d}{phase}.gdf')
        if not os.path.exists(fname_gdf):
             try:
                 sfreq = 250
                 ch_names = [f'EEG-{i}' for i in range(22)] + ['EOG-left', 'EOG-central', 'EOG-right']
                 ch_types = ['eeg'] * 22 + ['eog'] * 3
                 info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
                 dummy_data = np.random.randn(len(ch_names), sfreq * 10) # Longer dummy data
                 dummy_raw = mne.io.RawArray(dummy_data, info)
                 # Add multiple annotations for epoching
                 onsets = np.linspace(0.5, 8.5, 9) # Example onsets
                 durations = [0.1] * len(onsets)
                 descriptions = ['783'] * len(onsets) # Use unknown event ID
                 if phase == 'T':
                     # Ensure enough events for L/R
                     num_events = len(onsets)
                     descriptions = ['769'] * (num_events // 2) + ['770'] * (num_events - num_events // 2)
                 annotations = mne.Annotations(onsets, durations, descriptions)
                 dummy_raw.set_annotations(annotations)
                 fname_fif = fname_gdf.replace('.gdf', '.fif')
                 dummy_raw.save(fname_fif, overwrite=True, verbose=False)
             except Exception as e:
                 print(f"Could not create dummy file {fname_gdf}: {e}. Data loading might fail.")

    fname_mat = os.path.join(true_labels_dir, f'A{subj:02d}E.mat')
    if not os.path.exists(fname_mat):
        dummy_labels = {'classlabel': np.random.randint(1, 5, size=(288, 1))} # Random labels 1-4
        from scipy.io import savemat
        savemat(fname_mat, dummy_labels)

# Parameters matching paper
sfreq = 250
tmin = 0.5       # Epoch start 0.5s post-cue
tmax = 3.5       # Epoch end 2.5s post-cue (2 second duration)
n_samples = int((tmax - tmin) * sfreq) # Expected samples per epoch

# Filter parameters matching paper
fmin = 8.0
fmax = 30.0
filter_order = 5

train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, n_samples) per subject.
train_active_y = []         # List to hold event labels per subject.

print("\n--- Loading Training Data (Paper Preprocessing) ---")
# Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
for subj in range(1, 10):
    filename_gdf = os.path.join(data_dir, f'A{subj:02d}T.gdf')
    filename_fif = filename_gdf.replace('.gdf', '.fif') # Use FIF if GDF doesn't exist

    try:
        # Load raw data
        if os.path.exists(filename_gdf):
             train_raw = mne.io.read_raw_gdf(filename_gdf, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        elif os.path.exists(filename_fif):
             train_raw = mne.io.read_raw_fif(filename_fif, preload=True, verbose=False)
        else:
            print(f"Skipping subject {subj}: No training file found ({filename_gdf} or {filename_fif})")
            train_active_X.append(np.array([]))
            train_active_y.append(np.array([]))
            continue

        # Retain only EEG channels
        train_raw.pick_types(eeg=True, eog=False)

        # Apply 5th order Butterworth bandpass filter (8-30 Hz) - matches paper
        # Using MNE's recommended 'sos' output for IIR stability
        train_raw.filter(l_freq=fmin, h_freq=fmax,
                         method='iir',
                         iir_params=dict(order=filter_order, ftype='butter', output='sos'),
                         skip_by_annotation='edge', verbose=False)

        # Extract events corresponding only to the Left/Right types.
        train_active_events, _ = mne.events_from_annotations(train_raw, event_id=active_all_event_ids, verbose=False)

        # Create epochs from tmin to tmax.
        train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=active_all_event_ids,
                                         tmin=tmin, tmax=tmax, baseline=None, # No baseline correction needed after filtering
                                         preload=True, verbose=False, proj=False)

        # Get the epoch data (num_epochs x 22 channels x n_samples). Convert to microvolts.
        train_active_data = train_active_epochs.get_data() * 1e6

        # Verify shape
        print(f"Subject {subj}: Filtered Epoch data shape {train_active_data.shape}")
        if train_active_data.shape[-1] != n_samples:
             print(f"Warning: Subject {subj} train epoch samples ({train_active_data.shape[-1]}) != expected ({n_samples}). Check tmin/tmax/sfreq.")


        # Append the processed data, labels.
        train_active_X.append(train_active_data)
        train_active_y.append(train_active_epochs.events[:, 2])  # Event code (769, 770)

    except Exception as e:
        print(f"Error loading training data for subject {subj}: {e}")
        import traceback
        traceback.print_exc()
        train_active_X.append(np.array([]))
        train_active_y.append(np.array([]))


print(f"\nLoaded training data for {len(train_active_X)} subjects.")

eval_active_X = []
eval_active_y = []

print("\n--- Loading Evaluation Data (Paper Preprocessing) ---")
# Loop over subjects. Assume files are named "A01E.gdf", ..., "A09E.gdf".
for subj in range(1, 10):
    filename_gdf = os.path.join(data_dir, f'A{subj:02d}E.gdf')
    filename_fif = filename_gdf.replace('.gdf', '.fif') # Use FIF if GDF doesn't exist
    mat_filename = os.path.join(true_labels_dir, f'A{subj:02d}E.mat')

    try:
        # Load true labels
        mat_data = loadmat(mat_filename)
        true_y_all = np.array(mat_data['classlabel'], dtype=np.int64).reshape(-1,) + 768 # Map 1..4 to 769..772

        # Load raw data
        if os.path.exists(filename_gdf):
            eval_raw = mne.io.read_raw_gdf(filename_gdf, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        elif os.path.exists(filename_fif):
            eval_raw = mne.io.read_raw_fif(filename_fif, preload=True, verbose=False)
        else:
             print(f"Skipping subject {subj}: No evaluation file found ({filename_gdf} or {filename_fif})")
             eval_active_X.append(np.array([]))
             eval_active_y.append(np.array([]))
             continue

        # Retain only EEG channels
        eval_raw.pick_types(eeg=True, eog=False)

        # Apply 5th order Butterworth bandpass filter (8-30 Hz) - matches paper
        eval_raw.filter(l_freq=fmin, h_freq=fmax,
                        method='iir',
                        iir_params=dict(order=filter_order, ftype='butter', output='sos'),
                        skip_by_annotation='edge', verbose=False)

        # Extract *all* trial events using the 'unknown' ID (783)
        eval_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id, verbose=False)

        # Create epochs around all cues
        eval_epochs = mne.Epochs(eval_raw, eval_events, event_id=unknown_event_id,
                                 tmin=tmin, tmax=tmax, baseline=None,
                                 preload=True, verbose=False, proj=False)

        # Get epoch data and convert to microvolts
        eval_data = eval_epochs.get_data() * 1e6

        # Verify shape
        print(f"Subject {subj}: Filtered Epoch data shape {eval_data.shape}")
        if eval_data.shape[0] != len(true_y_all):
             print(f"Warning: Mismatch between epoch count ({eval_data.shape[0]}) and true labels ({len(true_y_all)}) for subject {subj}. Skipping.")
             eval_active_X.append(np.array([]))
             eval_active_y.append(np.array([]))
             continue
        if eval_data.shape[-1] != n_samples:
             print(f"Warning: Subject {subj} eval epoch samples ({eval_data.shape[-1]}) != expected ({n_samples}). Check tmin/tmax/sfreq.")

        # Append the processed data and labels.
        eval_active_X.append(eval_data)
        eval_active_y.append(true_y_all)  # Append all labels first

    except Exception as e:
        print(f"Error loading evaluation data for subject {subj}: {e}")
        import traceback
        traceback.print_exc()
        eval_active_X.append(np.array([]))
        eval_active_y.append(np.array([]))


print(f"\nLoaded evaluation data for {len(eval_active_X)} subjects.")

# # Filter evaluation data for Left/Right classes AFTER loading all subjects
# print("\n--- Filtering Evaluation Data for L/R Classes ---")
# eval_active_X_lr = []
# eval_active_y_lr = []
# for x, y in zip(eval_active_X, eval_active_y):
#      if x.size > 0 and y.size > 0:
#          lr_indices = np.isin(y, [769, 770])
#          if np.sum(lr_indices) > 0:
#              eval_active_X_lr.append(x[lr_indices])
#              eval_active_y_lr.append(y[lr_indices])
#              print(f"Filtered L/R trials shape: {x[lr_indices].shape}")
#          else:
#              print("Warning: No L/R trials found in evaluation set for one subject.")
#              eval_active_X_lr.append(np.array([])) # Keep list aligned
#              eval_active_y_lr.append(np.array([]))
#      else:
#          eval_active_X_lr.append(np.array([])) # Keep list aligned
#          eval_active_y_lr.append(np.array([]))


# Encode labels to 0 and 1
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

print("\n--- Encoding Labels ---")
train_active_y_encoded = encode_labels(train_active_y)
eval_active_y_encoded = encode_labels(eval_active_y)

# --- MEKT Components (Defined after data loading) ---

# 1. Covariance Estimation
cov_estimator = Covariances(estimator='lwf') # Ledoit-Wolf shrinkage

# 2. Tangent Space Mapping (centered at Identity)
ts_mapper = TangentSpace(metric='riemann', tsupdate=False)

# 3. JDA Implementation (adapted from https://github.com/chamwen/MEKT/blob/master/MEKT/JDA.py)
def kernel(ker, X1, X2, gamma):
    """Calculates the kernel matrix."""
    K = None
    if ker == 'linear':
        if X2 is not None: K = pairwise_kernels(X1, X2, metric='linear')
        else: K = pairwise_kernels(X1, metric='linear')
    elif ker == 'rbf':
        if X2 is not None: K = pairwise_kernels(X1, X2, metric='rbf', gamma=gamma)
        else: K = pairwise_kernels(X1, metric='rbf', gamma=gamma)
    elif ker == 'poly':
         if X2 is not None: K = pairwise_kernels(X1, X2, metric='poly', degree=2)
         else: K = pairwise_kernels(X1, metric='poly', degree=2)
    return K

def jda(Xs, Xt, Ys, Yt_pseudo, k=100, lambda_reg=1.0, ker='linear', gamma=1.0):
    """Joint Distribution Adaptation (JDA) - Implementation unchanged"""
    X = np.vstack((Xs, Xt))
    n, d = X.shape
    ns, nt = Xs.shape[0], Xt.shape[0]
    unique_Ys = np.unique(Ys)
    if not np.all(np.isin(unique_Ys, [0, 1, 2, 3])):
         print(f"Error: Source labels Ys contain unexpected values: {unique_Ys}. Expected 0 and 1.")
         return None, None, None
    C = len(unique_Ys)
    # MMD matrix M0
    e_s = np.ones((ns, 1)) / ns
    e_t = -np.ones((nt, 1)) / nt
    e = np.vstack((e_s, e_t))
    M0 = e @ e.T
    # Conditional MMD matrix Mc
    N = 0
    valid_pseudo = False
    if Yt_pseudo is not None and len(Yt_pseudo) == nt:
        unique_Yt_pseudo = np.unique(Yt_pseudo)
        if np.all(np.isin(unique_Yt_pseudo, [0, 1])) and len(unique_Yt_pseudo) >= 1:
             valid_pseudo = True
             C_pseudo = len(unique_Yt_pseudo)
             # print(f"Using {C_pseudo} classes from pseudo labels for conditional MMD.") # Less verbose
        # else: print(f"Warning: Pseudo target labels contain unexpected values: {unique_Yt_pseudo}. Using only marginal MMD.") # Less verbose
    # else: print("Warning: Invalid or missing pseudo target labels. Using only marginal MMD (M0).") # Less verbose

    if valid_pseudo:
        Mc = np.zeros((n, n))
        for c in unique_Ys:
            e_sc = np.zeros((ns, 1))
            e_tc = np.zeros((nt, 1))
            idx_sc = np.where(Ys == c)[0]
            idx_tc = np.where(Yt_pseudo == c)[0]
            count_sc = len(idx_sc)
            count_tc = len(idx_tc)
            if count_sc > 0: e_sc[idx_sc] = 1 / count_sc
            if count_tc > 0: e_tc[idx_tc] = -1 / count_tc
            ec = np.vstack((e_sc, e_tc))
            if count_sc > 0 and count_tc > 0:
                 Mc += ec @ ec.T
                 N += 1
        if N > 0:
             Mc = Mc / N
             # print(f"Conditional MMD matrix Mc constructed using {N} common classes.") # Less verbose
        else:
             # print("Warning: Conditional MMD matrix Mc is zero (no common classes found).") # Less verbose
             Mc = np.zeros((n, n))
    else:
        Mc = np.zeros((n, n))
    # Combine MMD
    M = M0 + Mc
    # Kernel matrix K
    K = kernel(ker, X, None, gamma)
    # Solve generalized eigenvalue problem
    H = np.eye(n) - (1 / n) * np.ones((n, n))
    if ker == 'linear':
        reg_eye = 1e-9 * np.eye(d)
        left_matrix = X.T @ M @ X + lambda_reg * np.eye(d)
        right_matrix = X.T @ H @ X + reg_eye
        left_matrix = (left_matrix + left_matrix.T) / 2
        right_matrix = (right_matrix + right_matrix.T) / 2
        try:
            eigvals, eigvecs = eigh(left_matrix, right_matrix)
            if np.any(np.isnan(eigvals)) or np.any(np.isinf(eigvals)): raise ValueError("NaN/Inf eigenvalues")
            idx = np.argsort(eigvals)
            A = eigvecs[:, idx[:k]].real
            Z = X @ A
            Zs = Z[:ns, :]
            Zt = Z[ns:, :]
            return Zs, Zt, A
        except (np.linalg.LinAlgError, ValueError) as e:
            print(f"Error solving linear JDA eigenvalue problem: {e}.")
            return None, None, None
    else: # Kernel JDA
        reg_eye_n = 1e-9 * np.eye(n)
        left_matrix = K @ M @ K.T + lambda_reg * np.eye(n)
        right_matrix = K @ H @ K.T + reg_eye_n
        left_matrix = (left_matrix + left_matrix.T) / 2
        right_matrix = (right_matrix + right_matrix.T) / 2
        try:
            eigvals, eigvecs = eigh(left_matrix, right_matrix)
            if np.any(np.isnan(eigvals)) or np.any(np.isinf(eigvals)): raise ValueError("NaN/Inf eigenvalues")
            idx = np.argsort(eigvals)
            A = eigvecs[:, idx[:k]].real
            Z = K @ A
            Zs = Z[:ns, :]
            Zt = Z[ns:, :]
            return Zs, Zt, A
        except (np.linalg.LinAlgError, ValueError) as e:
             print(f"Error solving kernel JDA eigenvalue problem: {e}.")
             return None, None, None


# 4. Classifier
classifier = SVC(kernel='linear', probability=True, C=1.0) # Linear SVM is common

# --- Cross-Subject Evaluation Loop ---
accuracies = []
n_subjects_loaded = 9
k_dim = 50
lambda_param = 1.0
kernel_type = 'linear'
gamma_param = 1.0

print(f"\n--- Starting Cross-Subject MEKT-MTS Evaluation (Target Dim k={k_dim}) ---")

for target_subj_idx in range(n_subjects_loaded):
    print(f"\n--- Target Subject Index: {target_subj_idx} (Subject {target_subj_idx + 1}) ---")

    # Prepare Data (Checks unchanged)
    if target_subj_idx >= len(eval_active_X) or \
       eval_active_X[target_subj_idx].size == 0 or \
       target_subj_idx >= len(eval_active_y_encoded) or \
       eval_active_y_encoded[target_subj_idx].size == 0:
        print(f"Skipping target subject {target_subj_idx + 1}: Missing valid L/R evaluation data.")
        continue
    Xt_raw = eval_active_X[target_subj_idx]
    Yt_true = eval_active_y_encoded[target_subj_idx]

    Xs_raw_list = [train_active_X[i] for i in range(n_subjects_loaded)
                   if i != target_subj_idx and i < len(train_active_X) and train_active_X[i].size > 0]
    Ys_list = [train_active_y_encoded[i] for i in range(n_subjects_loaded)
               if i != target_subj_idx and i < len(train_active_y_encoded) and train_active_y_encoded[i].size > 0]

    if not Xs_raw_list or not Ys_list or len(Xs_raw_list) != len(Ys_list):
        print(f"Skipping target subject {target_subj_idx + 1}: No valid source data.")
        continue

    try:
        Xs_raw = np.concatenate(Xs_raw_list, axis=0)
        Ys = np.concatenate(Ys_list, axis=0)
    except ValueError as e:
        print(f"Error concatenating source data for target {target_subj_idx + 1}: {e}")
        continue

    print(f"Target Subj {target_subj_idx + 1}: Source shape: {Xs_raw.shape}, Target shape: {Xt_raw.shape}")
    print(f"Target Subj {target_subj_idx + 1}: Source labels shape: {Ys.shape}, Target labels shape: {Yt_true.shape}")

    if Xs_raw.size == 0 or Xt_raw.size == 0 or Ys.size == 0 or Yt_true.size == 0:
         print(f"Skipping target subject {target_subj_idx + 1}: Empty data after selection.")
         continue

    try:
        # print(f"Target Subj {target_subj_idx + 1}: Source label distribution: {np.bincount(Ys)}") # Less verbose
        # print(f"Target Subj {target_subj_idx + 1}: Target label distribution: {np.bincount(Yt_true)}") # Less verbose
        if len(np.unique(Ys)) < 2:
             print(f"Warning: Source data for target {target_subj_idx + 1} has only one class. Skipping.")
             continue
    except ValueError as e:
         print(f"Warning: Could not calculate label distribution: {e}")
         continue

    # Feature Extraction Pipeline (Unchanged logic, uses data preprocessed above)
    try:
        # 1. Compute Covariances
        covs_s = cov_estimator.fit_transform(Xs_raw)
        covs_t = cov_estimator.transform(Xt_raw)

        # 2. Tangent Space Mapping
        Xs_ts = ts_mapper.fit_transform(covs_s)
        Xt_ts = ts_mapper.transform(covs_t)

        # print(f"Target Subj {target_subj_idx + 1}: Tangent Space features shape: Source={Xs_ts.shape}, Target={Xt_ts.shape}") # Less verbose

        if np.any(np.isnan(Xs_ts)) or np.any(np.isinf(Xs_ts)) or \
           np.any(np.isnan(Xt_ts)) or np.any(np.isinf(Xt_ts)):
            print(f"Error: NaN/Inf in Tangent Space features for target {target_subj_idx + 1}. Skipping.")
            continue

        # Initial Pseudo-Labeling
        # print("Training initial classifier on source tangent space data...") # Less verbose
        initial_clf = SVC(kernel='linear', C=1.0)
        initial_clf.fit(Xs_ts, Ys)
        Yt_pseudo = initial_clf.predict(Xt_ts)
        # print(f"Target Subj {target_subj_idx + 1}: Initial pseudo-label accuracy: {accuracy_score(Yt_true, Yt_pseudo):.4f}") # Less verbose

        # Apply JDA
        # print("Applying Joint Distribution Adaptation (JDA)...") # Less verbose
        Zs, Zt, A = jda(Xs_ts, Xt_ts, Ys, Yt_pseudo, k=k_dim, lambda_reg=lambda_param, ker=kernel_type, gamma=gamma_param)

        if Zs is None or Zt is None or A is None:
             print(f"JDA failed for target subject {target_subj_idx + 1}. Skipping.")
             continue
        if np.any(np.isnan(Zs)) or np.any(np.isinf(Zs)) or \
           np.any(np.isnan(Zt)) or np.any(np.isinf(Zt)):
            print(f"Error: NaN/Inf in JDA-transformed features for target {target_subj_idx + 1}. Skipping.")
            continue

        # print(f"Target Subj {target_subj_idx + 1}: Transformed features shape: Source={Zs.shape}, Target={Zt.shape}") # Less verbose

        # Train Classifier
        # print("Training final classifier on JDA-transformed source data...") # Less verbose
        classifier.fit(Zs, Ys)

        # Evaluate
        Yt_pred = classifier.predict(Zt)

        # Calculate Accuracy
        acc = accuracy_score(Yt_true, Yt_pred)
        accuracies.append(acc)
        print(f"--- Target Subject {target_subj_idx + 1} Accuracy: {acc:.4f} ---")

    except Exception as e:
        print(f"Error processing target subject {target_subj_idx + 1}: {e}")
        import traceback
        traceback.print_exc()

# --- Final Results ---
if accuracies:
    mean_accuracy = np.mean(accuracies)
    std_accuracy = np.std(accuracies)
    print("\n--- Overall Results ---")
    print(f"Individual Subject Accuracies: {[f'{a:.4f}' for a in accuracies]}")
    print(f"Mean Accuracy ({len(accuracies)} subjects): {mean_accuracy:.4f}")
    print(f"Standard Deviation: {std_accuracy:.4f}")
else:
    print("\nNo successful evaluations completed. Please check data loading and JDA steps.")



NOTE: Using placeholder directories: /home/vishwa/eeg_tl/Recreating papers/BCICIV_2a, /home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels
Please ensure the actual BCI Competition IV 2a GDF and true label MAT files are present in these locations.

--- Loading Training Data (Paper Preprocessing) ---
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Subject 1: Filtered Epoch data shape (288, 22, 751)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Subject 2: Filtered Epoch data shape (288, 22, 751)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Subject 3: Filtered Epoch data shape (288, 22, 751)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Subject 4: Filtered Epoch data shape (288, 22, 751)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Subject 5: Filtered Epoch data shape (288, 22, 751)
NOTE: pick_types() is a legacy function. New cod